# Chuyên đề 06 · VQ-VAE và token hóa ảnh: dựng VQ-VAE trên Fashion-MNIST, từ bảng mã đến sinh ảnh mới

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd06-vq-vae.ipynb)

Notebook đi kèm [Chuyên đề 06 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vq-vae/). Bài 19 nén một ảnh thành vài số *thực*. Mô hình
ngôn ngữ ở Bài 14 và 17 lại chỉ đọc và viết **token**: những số nguyên trong một bảng từ vựng. **VQ-VAE**
([van den Oord, Vinyals & Kavukcuoglu, 2017](https://arxiv.org/abs/1711.00937)) nối hai thế giới ấy: bộ mã hóa biến
ảnh thành một lưới vectơ, mỗi vectơ được thay bằng chỉ số của vectơ gần nhất trong một **bảng mã** (*codebook*),
và bộ giải mã dựng lại ảnh từ lưới chỉ số đó. Ảnh thành một chuỗi token, nên một Transformer như ở Bài 17 học được
cách viết ra ảnh mới, token sau token. VQGAN, MaskGIT, Parti, Chameleon và các bộ token hóa âm thanh của Chuyên đề 05
đều xây trên ý này.

Ta dựng VQ-VAE từ đầu trên Fashion-MNIST, ở kích thước chạy được trên CPU, kiểm chứng các nhận định của bài báo và
của những bài theo sau trên 3 seed, rồi sinh quần áo mới từ token.

**Bạn sẽ làm:**
1. Đếm bit: một ảnh 28 × 28 thành 49 token, mỗi token 7 bit.
2. Tự viết bước lượng tử hóa, mẹo *straight-through* và ba số hạng mất mát, rồi hỏi autograd xem số hạng nào dạy
   phần nào của mạng.
3. Thấy bảng mã "chết" (*codebook collapse*) ngay từ lúc khởi tạo, và sửa nó.
4. Huấn luyện VQ-VAE trực tiếp, rồi xem mỗi token mang thông tin gì.
5. Kiểm chứng sáu nhận định về VQ-VAE bằng kết quả 3 seed.
6. Học một prior Transformer nhỏ trên token, rồi sinh và điền tiếp ảnh.

**Cần biết trước:** [Bài 11 · CNN](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/cnn/),
[Bài 14 · Token & embedding](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/),
[Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) và
[Bài 19 · Autoencoder & VAE](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/).
**Thời gian chạy:** khoảng 1 phút trên CPU, kể cả tải Fashion-MNIST (~30 MB) lần đầu.

Notebook dùng **PyTorch**, như các chuyên đề khác. Phần huấn luyện trực tiếp chạy 400 bước đầu của mô hình chuẩn
(seed 0); các con số đầy đủ (3.000 bước, 3 seed, 22 cấu hình) do script `reel/vqvae/sweep.py` của khóa học tính sẵn,
cùng mã và cùng seed, và được nạp ở các mục sau.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

`torch.use_deterministic_algorithms(True)` buộc PyTorch dùng các phép tính cho cùng kết quả ở mỗi lần chạy. Không có
nó, gradient của bảng mã được cộng dồn bởi nhiều luồng theo thứ tự thay đổi giữa các lần chạy, và hai lần huấn luyện
cùng seed đã lệch nhau sau vài chục bước.

In [ ]:
import base64
import gzip
import hashlib
import json
import math
import os
import urllib.request
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=1):
    return vn(100 * p, digits) + "%"


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 3,2·10⁻⁶ (0 in ra 0)."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    return f"{vn(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.001 <= v <= 10000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Nhãn trục kiểu Việt Nam; gọi ngay trước plt.show(). Trục log ngắn lấy vạch 1–2–5, trục log dài lấy lũy thừa của 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def show_row(ax, imgs, title=None):
    """Vẽ một hàng ảnh 28 × 28 (nét sáng trên nền tối, như dữ liệu)."""
    ax.imshow(torch.cat(list(imgs.reshape(-1, 28, 28)), 1).numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_ylabel(title, rotation=0, ha="right", va="center", fontsize=9)

## 2 · Dữ liệu: Fashion-MNIST

Fashion-MNIST ([Xiao, Rasul & Vollgraf, 2017](https://arxiv.org/abs/1708.07747), giấy phép MIT) có cùng khuôn với
MNIST: 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, ảnh xám 28 × 28, mười loại quần áo, giày dép và túi. Ta tải
bốn tệp gốc từ repo của bộ dữ liệu và kiểm tra MD5 (cùng mã torchvision dùng). Như mọi bài của khóa học, 55.000 ảnh
đầu để huấn luyện, 5.000 ảnh cuối của phần huấn luyện để kiểm định (chọn siêu tham số), và 10.000 ảnh kiểm tra chỉ
dùng để báo kết quả. Cùng ô này nạp kết quả của script và các trọng số đã huấn luyện sẵn.

In [ ]:
URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FILES = {  # mã MD5 chuẩn của Fashion-MNIST
    "train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d",
    "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe",
    "t10k-images-idx3-ubyte.gz": "bef4ecab320f06d8554ea6380940ec79",
    "t10k-labels-idx1-ubyte.gz": "bb300cfdad3c16e7a12a480ee83cd310",
}
DATA = Path("fashion")
DATA.mkdir(exist_ok=True)
for name, md5 in FILES.items():
    p = DATA / name
    if not p.exists():
        urllib.request.urlretrieve(URL + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


R = fetch_json("vqvae-results.json", "25864d01c150ecacaa3bc00b3ba3d5b5")
W = fetch_json("vqvae-weights.json", "cc3977cf197e6f111dc657b546160cb3")

In [ ]:
CLASSES = ["Áo thun", "Quần", "Áo len", "Váy", "Áo khoác", "Dép", "Áo sơ mi", "Giày thể thao", "Túi", "Bốt"]


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        y = np.frombuffer(f.read(), np.uint8, offset=8)
    return torch.from_numpy(x.astype(np.float32) / 255), torch.from_numpy(y.astype(np.int64))


X_all, y_all = load("train")
X, V, y_tr = X_all[:55000], X_all[55000:], y_all[:55000]
T, y_te = load("t10k")
print(f"huấn luyện {tuple(X.shape)} · kiểm định {tuple(V.shape)} · kiểm tra {tuple(T.shape)}")
print(f"điểm ảnh bằng đúng 0 (nền đen): {pct(float((X == 0).float().mean()))} · độ sáng trung bình {vn(X.mean(), 3)}")

fig, axes = plt.subplots(1, 10, figsize=(11, 1.6))
for c, ax in enumerate(axes):
    ax.imshow(T[int((y_te == c).nonzero()[0, 0]), 0], cmap="gray")
    ax.set_title(CLASSES[c], fontsize=8)
    ax.axis("off")
plt.show()

Mỗi ô là ảnh đầu tiên của một lớp trong tập kiểm tra, nét sáng trên nền đen như dữ liệu gốc; tên lớp ở phía trên.
Một nửa số điểm ảnh là nền đen tuyệt đối: ta sẽ gặp lại con số này khi xem các token được dùng nhiều nhất.

## 3 · Đếm bit

Ảnh gốc có 784 điểm ảnh, mỗi điểm 8 bit. VQ-VAE của ta thu nhỏ ảnh 4 lần mỗi chiều, còn một lưới 7 × 7 = 49 ô; mỗi
ô là chỉ số của một trong K = 128 mã, tức log₂ 128 = 7 bit. Để so sánh: một autoencoder liên tục cùng lưới, mỗi ô
một vectơ 16 số thực 32 bit như ở Bài 19, còn *tốn bit hơn ảnh gốc*. Số thực không phải là nén; token mới là nén.

In [ ]:
K, D = 128, 16
bits_raw = 28 * 28 * 8
bits_float = 7 * 7 * D * 32
bits_vq = 7 * 7 * math.log2(K)
print(f"ảnh gốc:                     {vn(bits_raw, 0):>7s} bit")
print(f"lưới 7 × 7 × {D} số float32:   {vn(bits_float, 0):>7s} bit")
print(f"49 token, mỗi token 1 trong {K}: {vn(bits_vq, 0):>7s} bit  (nhỏ hơn ảnh gốc {vn(bits_raw / bits_vq, 1)} lần)")
assert bits_vq == 343

## 4 · Lượng tử hóa vectơ và straight-through

Bộ mã hóa là một CNN nhỏ (Bài 11): hai tích chập bước 2 đưa 28 × 28 về 7 × 7, rồi một tích chập 1 × 1 cho mỗi ô
một vectơ $z_e \in \mathbb{R}^{16}$. **Lượng tử hóa** thay $z_e$ bằng mã gần nhất trong bảng $e_1, \dots, e_K$
(phương trình 1–2 của bài báo):

$$k = \arg\min_j \lVert z_e - e_j \rVert^2, \qquad z_q = e_k .$$

Khoảng cách được tính cho mọi cặp (vectơ, mã) cùng lúc bằng khai triển
$\lVert z \rVert^2 - 2\, z \cdot e + \lVert e \rVert^2$: phần giữa là một phép nhân ma trận. Chỉ số $k$ chính là
token. Bộ giải mã đối xứng với bộ mã hóa (tích chập chuyển vị) và kết thúc bằng sigmoid, cho điểm ảnh trong [0, 1].

**Straight-through.** $\arg\min$ không có đạo hàm, nên lúc lan truyền ngược ta *giả vờ* lượng tử hóa là hàm đồng
nhất: gradient ở đầu vào của bộ giải mã được chép nguyên về $z_e$. Trong code là `ze + (zq − ze).detach()`: giá trị
bằng `zq`, còn gradient chảy theo `ze`.

Ô dưới định nghĩa mô hình. Lớp `VQ` còn có hai thứ ta sẽ cần ở mục 5: cách khởi tạo bảng mã (`init`) và hàm
`refresh` chạy sau mỗi bước để thay các mã không ai dùng.

In [ ]:
GRID = 7


class VQ(nn.Module):
    """Bảng mã K × D: mã gần nhất, cập nhật bằng mất mát (eq. 3) hoặc EMA, khởi động lại mã chết."""

    def __init__(self, K=128, D=16, init="data", update="loss", restart=True, decay=0.99, eps=1e-5):
        super().__init__()
        if init == "randn":
            w = torch.randn(K, D)
        elif init == "uniform":
            w = torch.empty(K, D).uniform_(-1 / K, 1 / K)
        else:
            w = torch.zeros(K, D)  # điền bằng đầu ra của bộ mã hóa ở batch đầu (init_from)
        if update == "loss":
            self.book = nn.Parameter(w)
        else:
            self.register_buffer("book", w)
            self.register_buffer("ema_sum", w.clone())
        self.register_buffer("count", torch.ones(K))  # EMA số lần mỗi mã được chọn trong một batch
        self.K, self.update, self.restart, self.decay, self.eps = K, update, restart, decay, eps
        self.restarts = 0

    def nearest(self, ze):
        """ze: (N, D) → chỉ số mã gần nhất (khoảng cách bình phương, khai triển ‖z‖² − 2z·e + ‖e‖²)."""
        e = self.book
        return ((ze**2).sum(1, keepdim=True) - 2 * ze @ e.T + (e**2).sum(1)).argmin(1)

    def forward(self, ze):
        idx = self.nearest(ze.detach())
        return self.book[idx], idx

    @torch.no_grad()
    def init_from(self, ze, g):
        self.book.copy_(ze[torch.randperm(len(ze), generator=g)[: self.K]])
        if self.update == "ema":
            self.ema_sum.copy_(self.book)

    @torch.no_grad()
    def refresh(self, ze, idx, g):
        """Sau mỗi bước: đếm mã, cập nhật EMA (nếu dùng), thay mã chết bằng đầu ra ngẫu nhiên của batch."""
        n = torch.bincount(idx, minlength=self.K).float()
        self.count.mul_(self.decay).add_(n, alpha=1 - self.decay)
        if self.update == "ema":
            s = torch.zeros_like(self.ema_sum).index_add_(0, idx, ze)
            self.ema_sum.mul_(self.decay).add_(s, alpha=1 - self.decay)
            tot = self.count.sum()
            N = (self.count + self.eps) / (tot + self.K * self.eps) * tot  # làm trơn Laplace như Sonnet
            self.book.copy_(self.ema_sum / N[:, None])
        if self.restart:
            dead = self.count < 1
            k = int(dead.sum())
            if k:
                new = ze[torch.randint(len(ze), (k,), generator=g)]
                self.book[dead] = new
                if self.update == "ema":
                    self.ema_sum[dead] = new
                self.count[dead] = 1
                self.restarts += k


class VQVAE(nn.Module):
    def __init__(self, quant="vq", K=128, D=16, ch=32, init="data", update="loss", restart=True):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, ch // 2, 4, 2, 1), nn.ReLU(), nn.Conv2d(ch // 2, ch, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.ReLU(), nn.Conv2d(ch, D, 1),
        )  # fmt: skip
        self.dec = nn.Sequential(
            nn.Conv2d(D, ch, 3, 1, 1), nn.ReLU(), nn.ConvTranspose2d(ch, ch // 2, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(ch // 2, 1, 4, 2, 1),
        )  # fmt: skip
        self.quant, self.d = quant, D
        self.q = VQ(K, D, init, update, restart) if quant == "vq" else None  # "none": autoencoder liên tục

    def encode(self, x):
        """(B, 1, 28, 28) → z_e dạng (B·49, D), mỗi hàng là vectơ của một ô 4 × 4 trong lưới 7 × 7."""
        return self.enc(x).permute(0, 2, 3, 1).reshape(-1, self.d)

    def decode_z(self, z):
        return torch.sigmoid(self.dec(z.reshape(-1, GRID, GRID, self.d).permute(0, 3, 1, 2)))

    def decode(self, idx):
        """Token (B, 7, 7) → ảnh: tra bảng mã rồi giải mã."""
        return self.decode_z(self.q.book[idx.reshape(-1)])

    def forward(self, x):
        ze = self.encode(x)
        if self.q is None:
            return self.decode_z(ze), ze, ze, None
        zq, idx = self.q(ze)
        z_st = ze + (zq - ze).detach()  # straight-through: giá trị là zq, gradient chảy về ze
        return self.decode_z(z_st), ze, zq, idx.reshape(-1, GRID, GRID)


m0 = VQVAE(init="randn")
x8 = X[:8]
with torch.no_grad():
    ze = m0.encode(x8)
    d_full = torch.cdist(ze, m0.q.book) ** 2
print(f"8 ảnh → z_e {tuple(ze.shape)} (8 · 49 vectơ 16 chiều); bảng mã {tuple(m0.q.book.shape)}")
assert torch.equal(m0.q.nearest(ze), d_full.argmin(1)), "khai triển phải chọn đúng mã như khoảng cách trực tiếp"
n_par = sum(p.numel() for n, p in m0.named_parameters() if not n.startswith("q."))
print(f"bộ mã hóa + bộ giải mã: {vn(n_par, 0)} tham số; bảng mã: {vn(m0.q.book.numel(), 0)} số")

Bảng mã chỉ có 128 × 16 = 2.048 số, nhỏ hơn mạng rất nhiều. Còn một câu hỏi: bảng mã học bằng cách nào?

### 🤔 Dự đoán trước
Huấn luyện chỉ với mất mát tái tạo $\lVert x - \hat x \rVert^2$. Gradient của nó đối với bảng mã lớn bao nhiêu?

In [ ]:
torch.manual_seed(0)
m0 = VQVAE(init="randn")
xr, ze, zq, idx = m0(x8)
F.mse_loss(xr, x8).backward()
g_book = 0.0 if m0.q.book.grad is None else float(m0.q.book.grad.abs().max())
print(f"max |∂ℒ_tái tạo/∂bảng mã| = {sci(g_book)}")
print(f"max |∂ℒ_tái tạo/∂W của lớp đầu bộ mã hóa| = {sci(float(m0.enc[0].weight.grad.abs().max()))}")
assert g_book == 0

**Đúng bằng 0.** Straight-through chép gradient từ đầu vào bộ giải mã sang $z_e$, *bỏ qua* bảng mã: giá trị
`zq` nằm trong `.detach()`. Bộ mã hóa và bộ giải mã học được, còn bảng mã đứng yên mãi. Vì thế bài báo thêm hai số
hạng, với $\mathrm{sg}[\cdot]$ là dừng gradient (*stop-gradient*: giá trị giữ nguyên, đạo hàm bằng 0):

$$\mathcal{L} = \lVert x - \hat x \rVert^2 + \lVert \mathrm{sg}[z_e] - e_k \rVert^2
+ \beta \lVert z_e - \mathrm{sg}[e_k] \rVert^2 .$$

Số hạng giữa (*mất mát mã*) kéo mã đã chọn về phía $z_e$, như một bước k-means. Số hạng cuối (*mất mát cam kết*,
*commitment*) kéo $z_e$ về phía mã đã chọn, để bộ mã hóa "cam kết" với một mã thay vì trôi đi; bài báo dùng
β = 0,25. Hai số hạng nhìn giống nhau nhưng **không đối xứng**: mỗi số hạng chỉ dạy một phía. Ô dưới hỏi autograd
xem mỗi số hạng gửi gradient tới đâu (in độ lớn gradient lớn nhất; 0 nghĩa là không có gradient).

In [ ]:
def loss_terms(model, x, beta):
    """Mất mát của bài báo (eq. 3, dạng trung bình): tái tạo + mã (‖sg[ze] − e‖²) + β · cam kết (‖ze − sg[e]‖²)."""
    xr, ze, zq, idx = model(x)
    rec = F.mse_loss(xr, x)
    if model.q is None:
        return rec, rec, ze, idx
    code = F.mse_loss(zq, ze.detach()) if model.q.update == "loss" else torch.zeros(())
    commit = F.mse_loss(ze, zq.detach())
    return rec + code + beta * commit, rec, ze, idx


def grad_size(t):
    return 0.0 if t.grad is None else float(t.grad.abs().max())


xr, ze, zq, idx = m0(x8)
terms = {"tái tạo": F.mse_loss(xr, x8), "mã": F.mse_loss(zq, ze.detach()), "cam kết": F.mse_loss(ze, zq.detach())}
print(f"{'số hạng':10s}{'bộ mã hóa':>12s}{'bảng mã':>12s}{'bộ giải mã':>12s}")
table = {}
for name, term in terms.items():
    m0.zero_grad()
    term.backward(retain_graph=True)
    table[name] = [grad_size(m0.enc[0].weight), grad_size(m0.q.book), grad_size(m0.dec[0].weight)]
    print(f"{name:10s}" + "".join(f"{sci(v):>12s}" for v in table[name]))
assert table["tái tạo"][1] == 0 and table["mã"][0] == table["mã"][2] == 0
assert table["cam kết"][1] == table["cam kết"][2] == 0 and table["cam kết"][0] > 0
m0.zero_grad()
terms["mã"].backward()
moved = int((m0.q.book.grad.abs().sum(1) > 0).sum())
print(f"mất mát mã chỉ chạm {moved} trong {K} mã: những mã được ít nhất một vectơ chọn")
assert moved == len(torch.unique(idx))

Bảng trên đúng như phương trình: tái tạo dạy bộ mã hóa (qua straight-through) và bộ giải mã; mất mát mã chỉ dạy
bảng mã; cam kết chỉ dạy bộ mã hóa. Dòng cuối là mầm mống của rắc rối ở mục sau: **chỉ những mã được chọn mới nhận
gradient**, và với bảng mã khởi tạo N(0, 1) này, cả 392 vectơ của 8 ảnh dồn vào một nắm mã (dòng cuối). Một mã
không bao giờ là mã gần nhất của vectơ nào thì không bao giờ học, và không bao giờ được dùng.

Bài báo cũng có một biến thể không cần mất mát mã (Phụ lục A.1, không dùng trong các thí nghiệm của bài): cập nhật
mỗi mã bằng trung bình trượt (EMA, hệ số γ = 0,99) của các vectơ đã chọn nó, tức k-means trực tuyến. Lớp `VQ` ở trên
có cả hai (`update="loss"` hoặc `"ema"`); mục 8 so sánh chúng.

## 5 · Bảng mã chết ngay từ lúc khởi tạo

Nhiều bản cài đặt khởi tạo bảng mã bằng $\mathcal{N}(0, 1)$ (ví dụ `zalandoresearch/pytorch-vq-vae` cho bản EMA,
`rosinality/vq-vae-2-pytorch`), số khác bằng phân phối đều $U(-1/K, 1/K)$ (taming-transformers, bản không EMA của
Zalando).

### 🤔 Dự đoán trước
Bảng 128 mã khởi tạo $\mathcal{N}(0, 1)$, mạng chưa học. Đưa 64 ảnh qua bộ mã hóa: 64 · 49 = 3.136 vectơ. Chúng chọn
bao nhiêu mã khác nhau?

In [ ]:
torch.manual_seed(0)
m0 = VQVAE(init="randn")
g = torch.Generator().manual_seed(0)
xb = X[torch.randint(len(X), (64,), generator=g)]
with torch.no_grad():
    ze = m0.encode(xb)
    books = {
        "N(0, 1)": m0.q.book.clone(),
        "U(−1/K, 1/K)": torch.empty(K, D).uniform_(-1 / K, 1 / K),
        "từ dữ liệu": ze[torch.randperm(len(ze), generator=g)[:K]],
    }
    print(f"{vn(len(ze), 0)} vectơ z_e, độ dài trung bình lúc khởi tạo {vn(ze.norm(dim=1).mean(), 3)}")
    used0 = {}
    for name, book in books.items():
        m0.q.book.data.copy_(book)
        used0[name] = len(torch.unique(m0.q.nearest(ze)))
        print(
            f"bảng mã {name:13s} độ dài mã ~{vn(book.norm(dim=1).mean(), 3):>6s} → {used0[name]:3d} / {K} mã được chọn"
        )
assert used0["N(0, 1)"] <= 5 and used0["từ dữ liệu"] > 100

Chỉ một nắm mã. Lúc khởi tạo, các vectơ $z_e$ ngắn (độ dài khoảng 0,4) và chụm gần gốc tọa độ, còn các mã
$\mathcal{N}(0, 1)$ 16 chiều dài khoảng $\sqrt{16} = 4$, nằm rải rác trên một mặt cầu xa. Với gần như mọi $z_e$, mã gần
nhất là cùng một hai mã tình cờ nằm gần gốc nhất. Các mã còn lại không được chọn, nên không nhận gradient (mục 4),
nên mãi không được chọn. Bảng mã đều $U(\pm 1/K)$ ngược lại: mọi mã nằm sát gốc, ngắn hơn $z_e$ hàng chục lần, và
cũng chỉ vài mã tình cờ nằm về phía các $z_e$ được chọn.

Hai cách sửa thường dùng, cả hai đã viết sẵn trong lớp `VQ`:

- **Khởi tạo từ dữ liệu:** lấy K vectơ $z_e$ ngẫu nhiên của batch đầu làm bảng mã (Karpathy và SoundStream dùng
  k-means trên batch đầu; ta chỉ lấy mẫu ngẫu nhiên, giống bước khởi đầu của k-means).
- **Khởi động lại mã chết** (Jukebox, Dhariwal và cộng sự, 2020): `refresh` giữ một trung bình trượt số lần mỗi mã
  được chọn trong một batch; mã nào xuống dưới 1 thì được đặt lại bằng một vectơ $z_e$ ngẫu nhiên của batch hiện tại.

Lúc khởi tạo, cách "từ dữ liệu" đã dùng đủ 128 mã. Nhưng huấn luyện có tự sửa được một khởi đầu tồi không? Ô
dưới nạp kết quả của script: 3.000 bước, 3 seed cho mỗi cách, và vẽ số mã được dùng trên tập kiểm định theo bước.

In [ ]:
INIT_RUNS = {
    "randn": "N(0, 1), không khởi động lại",
    "uniform": "U(±1/K), không khởi động lại",
    "data": "từ dữ liệu, không khởi động lại",
    "randn_restart": "N(0, 1) + khởi động lại",
    "ref": "từ dữ liệu + khởi động lại (chuẩn)",
}
STYLE = {
    "randn": (ORANGE, "o", "-"),
    "uniform": (ORANGE, "s", "--"),
    "data": (GREY, "^", "-"),
    "randn_restart": (BLUE, "D", "--"),
    "ref": (GREEN, "o", "-"),
}


def seeds(name, key="mse", where="test"):
    return np.array([run[where][key] for run in R["runs"][name]], dtype=float)


def span(a, digits=4, f=vn):
    a = np.asarray(a, dtype=float)
    return f(a.min(), digits) if np.isclose(a.min(), a.max()) else f"{f(a.min(), digits)}–{f(a.max(), digits)}"


STEPS = R["steps"]
steps_axis = [int(s) for s in R["runs"]["ref"][0]["checks"]]
fig, ax = plt.subplots(figsize=(9.5, 3.6))
for name, label in INIT_RUNS.items():
    c, mk, ls = STYLE[name]
    u = np.array([[run["checks"][str(s)]["used"] for s in steps_axis] for run in R["runs"][name]])
    ax.fill_between(steps_axis, u.min(0), u.max(0), color=c, alpha=0.15)
    ax.plot(steps_axis, np.median(u, 0), marker=mk, ls=ls, color=c, label=label)
ax.set(xscale="log", xlabel="bước huấn luyện", ylabel="số mã được dùng (trên 128)", ylim=(0, 135))
ax.set_title("số mã được dùng trên tập kiểm định (trung vị và dải 3 seed)", fontsize=10)
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
vn_axes(fig)
plt.tight_layout()
plt.show()
AT = (100, 400, 1000, 2000, STEPS)
print(f"{'số mã được dùng (kiểm định), bước':38s}" + "".join(f"{vn(s, 0):>12s}" for s in AT) + f"{'MSE kiểm tra':>17s}")
for name, label in INIT_RUNS.items():
    used = [span([run["checks"][str(s)]["used"] for run in R["runs"][name]], 0) for s in AT]
    print(f"{label:38s}" + "".join(f"{u:>12s}" for u in used) + f"{span(seeds(name)):>17s}")

Ba điều đọc được từ hình và bảng:

- **N(0, 1) không tự sửa được.** Sau 3.000 bước vẫn chỉ vài mã được dùng (đường cam liền), và MSE gấp đôi mô hình
  chuẩn. Bảng mã còn đó, nhưng bộ mã hóa chỉ có vài "từ" để nói.
- **U(±1/K) tự sửa, nhưng chậm.** Khi huấn luyện làm $z_e$ dài ra, chúng lần lượt kéo các mã sát gốc ra theo, và số
  mã được dùng tăng dần (đường cam đứt). Ở bước 400 nó vẫn chưa dùng tới một phần ba bảng mã: một bài hướng dẫn chạy
  ngắn sẽ kết luận rằng cách khởi tạo này "gây sụp đổ", trong khi thật ra nó chỉ chậm.
- **Khởi động lại mã chết chữa được cả khởi đầu tồi nhất.** N(0, 1) cộng khởi động lại (đường xanh dương) dùng đủ
  128 mã từ bước 1.000, và MSE nằm trong dao động giữa các seed của mô hình chuẩn. Khởi tạo từ dữ liệu mà không
  khởi động lại thì khá tốt, nhưng vẫn để vài mã chết và MSE cao hơn.

## 6 · Huấn luyện VQ-VAE

Mô hình chuẩn: K = 128 mã, D = 16, β = 0,25, mất mát của bài báo, khởi tạo từ dữ liệu và khởi động lại mã chết,
Adam với batch 64. Tốc độ học η được script chọn trên tập kiểm định với seed 0, rồi dùng chung cho mọi cấu hình. Ta
chạy trực tiếp 400 bước đầu của đúng lần chạy seed 0 ấy; mỗi bước rút batch, khởi tạo và khởi động
lại mã từ cùng một bộ sinh số ngẫu nhiên có seed, nên con số phải khớp với script.

In [ ]:
REF = R["ref"]
LIVE = R["live"]
etas = ", ".join(f"{vn_tick(e)} → {vn(v, 5)}" for e, v in zip(R["eta"]["grid"], R["eta"]["val"], strict=True))
print(f"MSE kiểm định sau {vn(R['steps'], 0)} bước theo η: {etas}; chọn η = {vn_tick(REF['lr'])}")


def train(cfg, seed, steps, log=None):
    torch.manual_seed(seed)  # trọng số ban đầu
    g = torch.Generator().manual_seed(seed)  # batch, khởi tạo bảng mã, khởi động lại mã chết
    m = VQVAE(cfg["quant"], cfg["K"], cfg["D"], 32, cfg["init"], cfg["update"], cfg["restart"])
    opt = torch.optim.Adam(m.parameters(), cfg["lr"])
    bs = cfg["bs"]
    if m.q is not None and cfg["init"] == "data":
        with torch.no_grad():
            m.q.init_from(m.encode(X[torch.randint(len(X), (bs,), generator=g)]), g)
    for step in range(1, steps + 1):
        x = X[torch.randint(len(X), (bs,), generator=g)]
        loss, rec, ze, idx = loss_terms(m, x, cfg["beta"])
        opt.zero_grad()
        loss.backward()
        opt.step()
        if m.q is not None:
            m.q.refresh(ze.detach(), idx.reshape(-1), g)
        if log is not None and step % 50 == 0:
            log.append(rec.item())
    return m.eval()


@torch.no_grad()
def metrics(m, X, chunk=2000):
    """MSE trung bình mỗi điểm ảnh, PSNR, số mã được dùng, perplexity exp(H) của tần suất mã."""
    se, codes = [], []
    for i in range(0, len(X), chunk):
        x = X[i : i + chunk]
        xr, _, _, idx = m(x)
        se.append(((xr - x) ** 2).mean((1, 2, 3)))
        if idx is not None:
            codes.append(idx.reshape(-1))
    mse = float(torch.cat(se).mean())
    out = {"mse": mse, "psnr": 10 * math.log10(1 / mse)}
    if codes:
        p = torch.bincount(torch.cat(codes), minlength=m.q.K).double() / sum(len(c) for c in codes)
        nz = p[p > 0]
        out.update(used=int((p > 0).sum()), ppl=float(torch.exp(-(nz * nz.log()).sum())))
    return out


live_curve = []
live = train(REF, 0, LIVE, live_curve)
got = metrics(live, V)
want = R["runs"]["ref"][0]["checks"][str(LIVE)]
print(f"sau {LIVE} bước (kiểm định): MSE {vn(got['mse'], 5)} (script {vn(want['mse'], 5)}) · "
      f"{got['used']} mã được dùng (script {int(want['used'])}) · perplexity {vn(got['ppl'], 1)} (script {vn(want['ppl'], 1)}) · "
      f"{live.q.restarts} lần thay mã")  # fmt: skip
assert abs(got["mse"] / want["mse"] - 1) < 0.05 and abs(got["used"] - want["used"]) <= 4, "khác với script"
assert abs(got["ppl"] / want["ppl"] - 1) < 0.1, "khác với script"

**Perplexity** là $\exp(H)$, với $H$ là entropy của tần suất các mã: số mã "hiệu dụng". Nó bằng số mã được dùng
nếu mọi mã được dùng đều nhau, và nhỏ hơn nếu vài mã chiếm phần lớn.

Trên máy khác (Colab dùng CPU x86), các phép tính dấu phẩy động có thể làm tròn khác ở chữ số cuối, và một vectơ nằm
sát ranh giới giữa hai mã có thể chọn mã khác; vì thế ô trên so với script trong một dung sai nhỏ thay vì đòi bằng
tuyệt đối.

Hình dưới vẽ mất mát tái tạo trên batch huấn luyện của lần chạy vừa rồi chồng lên lần chạy 3.000 bước của script.
Sau đó ta nạp chính mô hình 3.000 bước ấy (lưu ở dạng float16) và đo trên tập kiểm tra.

In [ ]:
ref_curve = R["runs"]["ref"][0]["curve"]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(np.arange(1, len(ref_curve) + 1) * 50, ref_curve, color=GREY, label=f"script, {vn(STEPS, 0)} bước")
ax.plot(np.arange(1, len(live_curve) + 1) * 50, live_curve, "o", ms=4, color=BLUE, label=f"chạy trực tiếp, {LIVE} bước")
ax.set(yscale="log", xlabel="bước", ylabel="MSE trên batch huấn luyện")
ax.legend()
vn_axes(fig)
plt.tight_layout()
plt.show()


def unpack(p):
    """Giải nén trọng số float16 (base64) thành state dict."""
    return {
        k: torch.tensor(np.frombuffer(base64.b64decode(v["f16"]), "<f2").astype(np.float32).reshape(v["shape"]))
        for k, v in p.items()
    }


vqm = VQVAE()
vqm.load_state_dict(unpack(W["vq"]))
vqm.eval()
aem = VQVAE("none")
aem.load_state_dict(unpack(W["ae"]))
aem.eval()
test = metrics(vqm, T)
want = R["runs"]["ref"][0]["test"]
print(f"mô hình {vn(STEPS, 0)} bước trên tập kiểm tra: MSE {vn(test['mse'], 5)} · PSNR {vn(test['psnr'], 2)} dB · "
      f"{test['used']} / {K} mã · perplexity {vn(test['ppl'], 1)}")  # fmt: skip
assert abs(test["mse"] / want["mse"] - 1) < 0.01 and abs(test["used"] - want["used"]) <= 1
assert abs(test["ppl"] / want["ppl"] - 1) < 0.02

Hình dưới đặt cạnh nhau: ảnh kiểm tra gốc, mô hình vừa chạy 400 bước, mô hình 3.000 bước, và một autoencoder liên
tục cùng lưới 7 × 7 × 16, cùng mạng, huấn luyện y hệt nhưng không lượng tử hóa (mục 8 so sánh hai loại này kỹ hơn).

In [ ]:
pick = torch.tensor([int((y_te == c).nonzero()[1, 0]) for c in range(10)])
with torch.no_grad():
    rows = [T[pick], live(T[pick])[0], vqm(T[pick])[0], aem(T[pick])[0]]
names = ["gốc", f"VQ-VAE {LIVE} bước", f"VQ-VAE {vn(STEPS, 0)} bước", "autoencoder liên tục"]
fig, axes = plt.subplots(4, 1, figsize=(10, 4.6))
for ax, row, name in zip(axes, rows, names, strict=True):
    show_row(ax, row, name)
plt.tight_layout()
plt.show()

Sau 400 bước, mô hình đã giữ được dáng của từng món (ống quần, tay áo, quai túi) nhưng còn mờ. Sau 3.000 bước, đường
viền và các chi tiết lớn như gấu áo len hay đế dép đã rõ hơn, nhưng họa tiết ca rô của chiếc áo sơ mi vẫn mất: 7 bit
cho một ô 4 × 4 không đủ chứa nó. Autoencoder liên tục cùng lưới giữ được cả họa tiết ấy.

## 7 · Một token mang thông tin gì?

Ô dưới mã hóa 10.000 ảnh kiểm tra thành token (chỉ cần bộ mã hóa và bước tìm mã gần nhất), rồi in lưới token của một
chiếc áo thun.

In [ ]:
@torch.no_grad()
def codes(m, X, chunk=5000):
    return torch.cat([m.q.nearest(m.encode(X[i : i + chunk])) for i in range(0, len(X), chunk)]).view(-1, GRID, GRID)


C_te = codes(vqm, T)
ex = int(pick[0])
print(f"ảnh kiểm tra #{ex} ({CLASSES[int(y_te[ex])]}): 49 token")
for row in C_te[ex].tolist():
    print("   " + " ".join(f"{c:3d}" for c in row))
flat_c = C_te.reshape(-1)
cnt = torch.bincount(flat_c, minlength=K)
cells = T[:, 0].unfold(1, 4, 4).unfold(2, 4, 4).mean((-1, -2))  # độ sáng trung bình của từng ô 4 × 4: (10000, 7, 7)
cell_mean = torch.zeros(K).index_add_(0, flat_c, cells.reshape(-1)) / cnt.clamp_min(1)
is_bg = (cell_mean < 0.02) & (cnt > 0)  # mã "nền": ô bên dưới gần như luôn đen
fg = ~is_bg & (cnt > 0)
bg = int(torch.where(is_bg, cnt, -1).argmax())  # mã nền hay gặp nhất
print(f"{int((cnt > 0).sum())} mã xuất hiện; {int(is_bg.sum())} trong số đó là mã 'nền' (ô bên dưới có độ sáng trung "
      f"bình < 0,02), chiếm tổng cộng {pct(float(cnt[is_bg].sum() / cnt.sum()))} số ô")  # fmt: skip
print(f"mã dùng nhiều nhất: {int(cnt.argmax())}, chiếm {pct(float(cnt.max() / cnt.sum()))} số ô")
here = sorted({c for c in C_te[ex].flatten().tolist() if is_bg[c]})
print(f"các mã nền có trong lưới của chiếc áo thun ở trên: {here}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={"width_ratios": [1, 2.2]})
axes[0].imshow(T[ex, 0], cmap="gray", extent=(0, 7, 7, 0))
for i in range(GRID):
    for j in range(GRID):
        c = int(C_te[ex, i, j])
        axes[0].text(j + 0.5, i + 0.5, str(c), ha="center", va="center", fontsize=6.5,
                     color="black" if T[ex, 0, 4 * i : 4 * i + 4, 4 * j : 4 * j + 4].mean() > 0.5 else "white")  # fmt: skip
axes[0].set(xticks=[], yticks=[], title="token trên từng ô 4 × 4")
order = torch.argsort(cnt, descending=True)
frac = 100 * cnt[order].numpy() / cnt.sum().item()
bgo = is_bg[order].numpy()
axes[1].bar(np.arange(K)[~bgo], frac[~bgo], color=BLUE, width=1.0, label="mã có nét")
axes[1].bar(np.arange(K)[bgo], frac[bgo], color=GREY, width=1.0, hatch="///", ec="white", lw=0, label="mã nền")
axes[1].legend(fontsize=8)
axes[1].set(yscale="log", xlabel="mã, xếp theo tần suất", ylabel="% số ô", title="tần suất 128 mã trên tập kiểm tra")
vn_axes(fig)
plt.tight_layout()
plt.show()

Lưới token bên trái cho thấy hai điều. Thứ nhất, *mép* của món đồ được mã hóa nhất quán: mép trái chiếc áo là cùng
một mã ở sáu hàng liền. Thứ hai, nền đen hai bên không dùng một mã chung mà dùng hơn chục mã khác nhau (dòng in cuối
ở ô trên). Cả mô hình có hàng chục mã "nền" như thế, chiếm hơn một phần ba số ô. Thủ phạm là chính cách sửa của
mục 5: một mã chết được đặt lại bằng một $z_e$ ngẫu nhiên của batch, mà một nửa số điểm ảnh là nền, nên mã mới hay
rơi vào vùng nền và chia nhau những ô vốn chỉ cần một mã. "Dùng đủ 128 mã" và perplexity khoảng 100 vì thế phóng
đại số "từ" thật sự khác nhau; Huh và cộng sự (2023) cũng cảnh báo perplexity cao có thể chỉ là dư thừa. Ở hình
phải, các mã nền (gạch chéo xám) nằm cả ở đầu bảng xếp hạng lẫn rải rác phía sau, và không mã nào áp đảo.

Vậy một mã có ứng với một loại đồ không, như một từ ứng với một khái niệm? Ô dưới đếm, với mỗi mã không phải nền,
tỉ lệ số lần nó xuất hiện trong loại đồ hay gặp nhất của nó, rồi vẽ sáu mã rải đều từ hay gặp đến ít gặp.

In [ ]:
M = torch.bincount(flat_c * 10 + y_te.repeat_interleave(GRID * GRID), minlength=K * 10).view(K, 10).float()
purity_fg = float(M[fg].max(1).values.sum() / M[fg].sum())
best = int(torch.argmax(torch.where(fg & (cnt > 500), M.max(1).values / M.sum(1).clamp_min(1), torch.tensor(0.0))))
print(f"với {int(fg.sum())} mã còn lại: trung bình {pct(purity_fg)} số lần xuất hiện của một mã rơi vào loại đồ hay gặp "
      f"nhất của nó (đoán bừa: 10%)")  # fmt: skip
share = M[best] / M[best].sum()
print(
    f"mã 'thuần' nhất trong các mã gặp trên 500 lần: mã {best}, {pct(float(share.max()))} là {CLASSES[int(share.argmax())]}"
)

ranked = [int(k) for k in order if fg[k]]
top = [ranked[r] for r in range(0, len(ranked), len(ranked) // 6)][:6]  # 6 mã có nét, rải đều theo tần suất
pad = F.pad(T[:, 0], (6, 6, 6, 6))
g = torch.Generator().manual_seed(0)
fig, axes = plt.subplots(len(top), 1, figsize=(8.5, 7))
for ax, k in zip(axes, top, strict=True):
    where = (C_te == k).nonzero()
    crops = torch.stack([pad[n, 4 * i : 4 * i + 16, 4 * j : 4 * j + 16] for n, i, j in where.tolist()])
    show = [crops.mean(0)] + list(crops[torch.randperm(len(crops), generator=g)[:8]])
    strip = torch.cat([F.pad(c, (1, 1, 1, 1), value=0.5) for c in show], 1)
    ax.imshow(strip.numpy(), cmap="gray", vmin=0, vmax=1)
    for s in range(len(show)):
        ax.add_patch(plt.Rectangle((18 * s + 6.5, 6.5), 4, 4, fill=False, ec=ORANGE, lw=1.2))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_ylabel(f"mã {k}\n({vn(cnt[k], 0)} ô)", rotation=0, ha="right", va="center", fontsize=9)
axes[0].set_title("ô đầu: trung bình mọi lần xuất hiện · 8 ô tiếp: lần xuất hiện ngẫu nhiên", fontsize=9)
plt.tight_layout()
plt.show()

Chỉ 16,5% (đoán bừa: 10%): một mã không nói "đây là áo len". Hình trên cho thấy mã nói gì. Mỗi hàng là một
mã: ô đầu là trung bình của vùng 16 × 16 điểm ảnh quanh mọi lần mã xuất hiện, tám ô sau là những lần xuất hiện ngẫu
nhiên, khung cam là ô 4 × 4 của token. Mỗi mã là một **mẫu cục bộ**: một mảng vải sáng đều, một mảng vải tối, một
mép dọc sáng bên này tối bên kia, một mép ngang. Cùng một mép có thể là mép của áo, của quần hay của túi. Ngay cả mã
"thuần" nhất trong các mã hay gặp cũng chỉ rơi vào một loại đồ khoảng một nửa số lần. Nghĩa "cả món đồ" không nằm
trong từng token mà trong cách 49 token xếp cạnh nhau, và đó là thứ prior ở mục 9 phải học.

## 8 · Kiểm chứng sáu nhận định

Mỗi nhận định dưới đây là một câu trong bài báo hoặc trong một tài liệu được trích dẫn nhiều. Script đã chạy mỗi cấu
hình 3.000 bước với 3 seed; ta đọc kết quả trên tập kiểm tra và kết luận *khớp*, *không khớp* hay *tùy điều kiện*,
như ở các Chuyên đề 02–05. Khi hai dải seed chồng lên nhau, khác biệt nằm trong dao động giữa các seed.

| | Nhận định | Nguồn |
|---|---|---|
| A | Kết quả không đổi với β từ 0,1 đến 2,0 | bài báo, mục 3.2 |
| B | Không có mất mát cam kết, $z_e$ lớn lên tùy ý | bài báo, mục 3.2 |
| C | Bản EMA học nhanh hơn bản dùng mất mát mã | chú thích của `VectorQuantizerEMA` trong Sonnet (DeepMind) |
| D | Lượng tử hóa làm tái tạo kém hơn mã liên tục | bài báo, mục 4.1 (4,67 so với 4,51 bit/chiều trên CIFAR-10); Rombach và cộng sự (2022), Bảng 8 |
| E | Bảng mã lớn hơn tái tạo tốt hơn, nhưng không có mẹo thì không dùng hết | Dhariwal và cộng sự (2020), mục 3.1; Yu và cộng sự (2021) |
| F | FSQ dùng gần hết bảng mã mà không cần mẹo; ở bảng mã nhỏ, VQ hơn một chút | Mentzer và cộng sự (2023) |

Cột ‖z_e‖ trong các bảng là độ dài trung bình của vectơ bộ mã hóa trên tập kiểm tra.

### A và B · Hệ số cam kết β

In [ ]:
LABEL = {
    "ref": "chuẩn (K 128, β 0,25)",
    "ema": "EMA thay cho mất mát mã",
    "ema_randn": "EMA, N(0, 1), không khởi động lại",
    "ae": "autoencoder liên tục",
    **{f"K{k}": f"K = {vn(k, 0)}" for k in (16, 32, 64, 256, 512, 1024)},
    "K512_norestart": "K = 512, không khởi động lại",
    "K1024_norestart": "K = 1.024, không khởi động lại",
    **{f"beta{b:g}": f"β = {vn(b, 2 if b == 0.25 else 1)}" for b in (0.0, 0.1, 1.0, 2.0)},
    "fsq128": "FSQ (8, 4, 4) = 128 mã",
    "fsq1000": "FSQ (8, 5, 5, 5) = 1.000 mã",
}


def table(names):
    print(f"{'cấu hình':36s}{'MSE kiểm tra':>18s}{'mã dùng':>14s}{'perplexity':>14s}{'‖z_e‖':>16s}")
    for n in names:
        has = "used" in R["runs"][n][0]["test"]
        used = span(seeds(n, "used"), 0) if has else "—"
        ppl = span(seeds(n, "ppl"), 1) if has else "—"
        z = seeds(n, "zNorm")
        print(f"{LABEL[n]:36s}{span(seeds(n)):>18s}{used:>14s}{ppl:>14s}{span(z, 2 if z.max() < 100 else 0):>16s}")


table(["ref", "beta0.1", "beta1", "beta2"])
for n in ("ref", "beta0"):
    end = [run["checks"][str(STEPS)] for run in R["runs"][n]]
    print(f"{LABEL[n]:24s} trên tập kiểm định (float32, bước {vn(STEPS, 0)}): MSE {span([e['mse'] for e in end])}, "
          f"‖z_e‖ {span([e['zNorm'] for e in end], 2 if n == 'ref' else 0)}")  # fmt: skip
bad = [run["seed"] for run in R["runs"]["beta0"] if run["test"]["mse"] is None]
print(f"β = 0, seed {', '.join(map(str, bad))}: bảng mã có giá trị vượt 65.504, số lớn nhất float16 lưu được, nên bản "
      "float16 của nó hỏng")  # fmt: skip

**A · gần như khớp.** β từ 0,1 đến 1 cho MSE trong dao động giữa các seed của β = 0,25; β = 2 kém hơn một chút
nhưng đều đặn ở cả 3 seed. Thứ β thật sự thay đổi là độ dài của $z_e$: cam kết càng mạnh, bộ mã hóa càng bị kéo về
sát các mã, và $z_e$ càng ngắn. Bài báo cũng lưu ý giá trị đúng của β phụ thuộc thang đo của mất mát tái tạo: ở đây
là MSE trung bình trên điểm ảnh trong [0, 1].

**B · khớp, rất rõ.** Bỏ hẳn cam kết (β = 0), độ dài $z_e$ tăng từ khoảng 2,5 lên hàng chục nghìn, và MSE tệ gấp
ba, bốn lần. Không gì giữ bộ mã hóa ở lại: mất mát mã kéo mã đuổi theo $z_e$, nhưng bộ mã hóa chạy nhanh hơn, đúng
như mục 3.2 của bài báo mô tả. Các mã lớn đến mức ở một seed, bản float16 mà script lưu không chứa nổi chúng, nên
dòng β = 0 được đọc trên tập kiểm định, từ các số script ghi lúc huấn luyện bằng float32.

### C · EMA thay cho mất mát mã

In [ ]:
ck = [s for s in steps_axis if s <= STEPS]
print("MSE kiểm định (trung vị 3 seed) theo bước")
print(f"{'bước':>8s}" + "".join(f"{s:>9d}" for s in ck))
for n in ("ref", "ema"):
    row = [np.median([run["checks"][str(s)]["mse"] for run in R["runs"][n]]) for s in ck]
    print(f"{n:>8s}" + "".join(f"{vn(v, 4):>9s}" for v in row))
table(["ref", "ema", "ema_randn"])
u = {s: span([run["checks"][str(s)]["used"] for run in R["runs"]["ema_randn"]], 0) for s in (1000, 2000)}
print(f"EMA, N(0, 1): {u[1000]} mã ở bước 1.000, {u[2000]} mã ở bước 2.000")
t_shrink = math.log(1 / 1e-5) / -math.log(0.99)
print(f"mã không ai chọn bắt đầu co về gốc khi 0,99ᵗ < ε = 10⁻⁵, tức sau t ≈ {vn(t_shrink, 0)} bước")

**C · không khớp ở đây.** Cùng khởi tạo từ dữ liệu và khởi động lại, hai cách cập nhật đi cùng một đường ở mọi bước,
và kết quả cuối nằm trong dao động giữa các seed. EMA khác ở một chỗ không ai quảng cáo: với N(0, 1) và *không* khởi
động lại, bản EMA tự hồi phục được vài chục mã, bản mất mát mã thì không (bảng ở mục 5: 3–6 mã ở bước 3.000). Lý do
nằm trong phép làm trơn Laplace của Sonnet: với một mã không ai chọn, cả tổng `ema_sum` lẫn bộ đếm `count` cùng co
theo $0{,}99^t$, nhưng mẫu số được cộng thêm ε = 10⁻⁵. Khi $0{,}99^t$ xuống dưới ε, mã ấy co dần về gốc tọa độ, đúng
nơi các $z_e$ nằm, và được chọn lại; số mã được dùng nhảy lên giữa bước 1.000 và 2.000, đúng lúc ô trên dự đoán. Đây
là tính chất của cách cài đặt này (bộ đếm bắt đầu bằng 1); các bản cài khác của cùng công thức xử lý mã chết khác
nhau.

### D · Lượng tử hóa và chất lượng tái tạo

Người ta hay nói ảnh của VQ-VAE "nét hơn" ảnh của VAE vì mã rời rạc. Thử trực tiếp.

### 🤔 Dự đoán trước
Cùng mạng, cùng lưới 7 × 7 × 16, cùng cách huấn luyện; một bên lượng tử hóa mỗi ô về 1 trong 128 mã, một bên giữ
nguyên 16 số thực. Bên nào dựng lại ảnh kiểm tra với MSE nhỏ hơn?

In [ ]:
table(["ae", "ref", "K1024"])
print(f"autoencoder liên tục / VQ-VAE chuẩn: MSE nhỏ hơn {vn(np.median(seeds('ref')) / np.median(seeds('ae')), 2)} lần")

**D · khớp.** Autoencoder liên tục dựng lại tốt hơn hẳn: MSE nhỏ hơn 2,38 lần so với VQ-VAE chuẩn, và vẫn nhỏ hơn
VQ-VAE với 1.024 mã. Lượng tử hóa là một nút thắt *thêm vào*, nó chỉ có thể làm mất thông tin. Bài báo cũng thấy vậy
(4,67 bit/chiều so với 4,51 của VAE liên tục trên CIFAR-10), và Latent Diffusion thấy bộ tự mã hóa điều chuẩn bằng VQ
dựng lại kém hơn một chút so với bản điều chuẩn bằng KL. Ảnh của VQGAN nét là nhờ mất mát cảm nhận và GAN (Esser và
cộng sự, 2021), không nhờ lượng tử hóa. Cái được của VQ là thứ khác: 343 bit thay vì 25.088, và một chuỗi token mà
một Transformer học được (mục 9).

### E · Kích thước bảng mã

Hình dưới đặt mọi cấu hình lên cùng một trục: số bit của một ảnh (49 · log₂ số mã) và MSE trên tập kiểm tra. Các
điểm FSQ được giải thích ở F ngay sau.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ks = (16, 32, 64, 128, 256, 512, 1024)
kname = {k: "ref" if k == 128 else f"K{k}" for k in ks}
for names, c, mk, dx, label in (
    ([kname[k] for k in ks], GREEN, "o", 0, "VQ, có khởi động lại"),
    (["K512_norestart", "K1024_norestart"], ORANGE, "s", 5, "VQ, không khởi động lại"),
    (["fsq128", "fsq1000"], BLUE, "D", -5, "FSQ (8, 4, 4) và (8, 5, 5, 5)"),
):
    size = [np.prod(R["grid"][n]["levels"]) if "fsq" in n else R["runs"][n][0]["cfg"]["K"] for n in names]
    bits = np.array([49 * math.log2(k) for k in size]) + dx  # dịch ngang vài bit cho các điểm khỏi chồng nhau
    med = np.array([np.median(seeds(n)) for n in names])
    err = [med - [seeds(n).min() for n in names], [seeds(n).max() for n in names] - med]
    ax.errorbar(bits, med, yerr=err, color=c, marker=mk, ls="-" if c == GREEN else "none", capsize=3, label=label)
ax.axhline(np.median(seeds("ae")), color=GREY, ls="--", label=f"autoencoder liên tục ({vn(bits_float, 0)} bit)")
ax.set_xticks([49 * math.log2(k) for k in ks], [f"{vn(49 * math.log2(k), 0)}\nK {vn(k, 0)}" for k in ks])
ax.set(xlabel="bit cho một ảnh (49 · log₂ số mã)", ylabel="MSE trên tập kiểm tra", ylim=(0, None))
ax.legend(fontsize=8)
vn_axes(fig)
plt.tight_layout()
plt.show()
table(["K16", "K64", "ref", "K256", "K512", "K1024", "K512_norestart", "K1024_norestart"])

**E · khớp.** Có khởi động lại, mọi K dùng gần hết bảng mã, và MSE giảm đều khi số bit tăng (đường xanh lá): gấp
đôi K là thêm 49 bit, và MSE giảm mỗi lần một ít. Ngay cả 1.024 mã (490 bit) vẫn còn xa autoencoder liên tục
(đường xám). Không khởi động lại (hình vuông cam), K = 512 và K = 1.024 bỏ phí phần lớn bảng mã, và MSE tệ hơn bản
có khởi động lại cùng K, ngang những bảng mã nhỏ hơn nhiều lần: K = 512 ngang K = 128–256, K = 1.024 ngang
K = 64–128. Dải seed của chúng cũng rộng hơn hẳn: bao nhiêu mã sống sót là chuyện may rủi.

### F · FSQ: bỏ hẳn bảng mã

FSQ (*finite scalar quantization*, Mentzer và cộng sự, 2023) thay bảng mã học được bằng một lưới cố định. Bộ mã hóa
chỉ xuất ra vài số mỗi ô (ở đây 3 hoặc 4 thay vì 16); mỗi số được chặn bằng tanh rồi làm tròn về một trong $L_i$ mức,
với straight-through cho phép làm tròn. Mức (8, 4, 4) cho 8 · 4 · 4 = 128 tổ hợp, đúng bằng K của mô hình chuẩn;
(8, 5, 5, 5) cho 1.000. Không có mất mát mã, không có cam kết, không có mã nào "chết" theo nghĩa của mục 5, vì mọi
điểm của lưới luôn nằm trong vùng $z_e$ có thể với tới. Bộ token hóa Cosmos của NVIDIA (2025) dùng FSQ.

In [ ]:
table(["ref", "fsq128", "K1024", "K1024_norestart", "fsq1000"])
for run in R["runs"]["fsq1000"]:
    u = [int(run["checks"][str(s)]["used"]) for s in (1000, 2000, STEPS)]
    print(
        f"FSQ 1.000 mã, seed {run['seed']}: {u[0]} → {u[1]} → {u[2]} mã được dùng ở bước 1.000 → 2.000 → {vn(STEPS, 0)}"
    )

**F · tùy điều kiện.** Ở 128 mã, FSQ dùng đủ cả 128 mà không cần mẹo nào, và VQ hơn nó một chút về MSE, đúng như
bài báo nói về các bảng mã nhỏ. Ở 1.000 mã, FSQ chỉ dùng từ một phần ba đến hai phần ba số mã sau 3.000 bước, chứ
không gần 100% như bài báo báo cáo; nó ngang VQ *không* khởi động lại, còn VQ có khởi động lại dùng đủ và dựng lại
tốt hơn. Bài báo đo trên những mô hình lớn hơn và học lâu hơn nhiều lần (MaskGIT, UViM); ở quy mô này, ưu thế "dùng
hết bảng mã" của FSQ chưa hiện ra, dù ở hai trong ba seed số mã nó dùng vẫn còn đang tăng ở bước cuối.

## 9 · Sinh ảnh mới: prior trên token

Bộ giải mã chỉ biến token thành ảnh; muốn *sinh* ảnh mới, ta cần biết lưới token nào là khả dĩ, tức một phân phối
$p(\text{token}_1, \dots, \text{token}_{49})$. Lúc huấn luyện, VQ-VAE coi phân phối này là đều: mọi lưới đều khả dĩ
như nhau. Khi đó số hạng KL trong ELBO của Bài 19 là hằng số, log K cho mỗi token, nên bị bỏ khỏi hàm mất mát (mục
3.1 của bài báo). Chỉ *sau khi* huấn luyện xong, bài báo mới học một prior tự hồi quy trên token (một PixelCNN, mục
3.3).

Ta dùng một Transformer nhân quả nhỏ như ở Bài 17: đọc lưới 7 × 7 theo từng hàng, như đọc một câu 49 từ, và dự
đoán mỗi token từ các token trước nó:

$$p(\text{token}_1, \dots, \text{token}_{49}) = \prod_{t=1}^{49} p(\text{token}_t \mid \text{token}_1, \dots,
\text{token}_{t-1}) .$$

Script đã huấn luyện nó trên token của 55.000 ảnh huấn luyện, η chọn trên tập kiểm định. Thước đo là **số bit cho
một ảnh**: $-\log_2 p(\text{token của ảnh})$, trung bình trên ảnh kiểm tra. Prior càng hiểu quần áo, càng ít bit;
đó cũng là độ dài trung bình nếu ta dùng prior ấy để nén ảnh (mã hóa số học). Ta so với ba prior đơn giản, cùng
được làm trơn bằng cách cộng 1 vào mọi tần suất: đều (log₂ 128 = 7 bit mỗi token), tần suất chung của từng mã, và
tần suất của từng mã *tại từng vị trí*. Ô dưới tính lại cả bốn trên 2.000 ảnh kiểm tra đầu và so với script.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.qkv, self.proj = nn.LayerNorm(d), nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.ln2, self.fc, self.out = nn.LayerNorm(d), nn.Linear(d, 4 * d), nn.Linear(4 * d, d)
        self.heads = heads

    def forward(self, h):
        B, T, d = h.shape
        q, k, v = self.qkv(self.ln1(h)).view(B, T, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # mỗi vị trí chỉ nhìn các token trước nó
        h = h + self.proj(a.transpose(1, 2).reshape(B, T, d))
        return h + self.out(F.gelu(self.fc(self.ln2(h))))


class Prior(nn.Module):
    """p(token₁ … token₄₉) = Π p(tokenₜ | các token trước), đọc lưới 7 × 7 theo hàng; token K là "bắt đầu"."""

    def __init__(self, K=128, d=128, layers=4, heads=4, T=GRID * GRID):
        super().__init__()
        self.tok, self.pos = nn.Embedding(K + 1, d), nn.Parameter(torch.randn(T, d) * 0.02)
        self.blocks = nn.ModuleList(Block(d, heads) for _ in range(layers))
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, K)
        self.K = K

    def forward(self, c):
        """c: (B, t) token đã có → logits (B, t, K) cho token kế tiếp ở mỗi vị trí (vị trí 0 đọc token bắt đầu)."""
        inp = torch.cat([torch.full_like(c[:, :1], self.K), c[:, :-1]], 1)
        h = self.tok(inp) + self.pos[: c.shape[1]]
        for b in self.blocks:
            h = b(h)
        return self.head(self.ln(h))

    def nll_bits(self, c, chunk=1000):
        """Trung bình −log₂ p(ảnh) trên các chuỗi c (n, 49): số bit cần để mã hóa một ảnh bằng prior này."""
        with torch.no_grad():
            tot = sum(
                float(F.cross_entropy(self(c[i : i + chunk]).transpose(1, 2), c[i : i + chunk], reduction="sum"))
                for i in range(0, len(c), chunk)
            )
        return tot / len(c) / math.log(2)


def baseline_bits(Ctr, Cte, K):
    """−log₂ p(ảnh) trên tập kiểm tra cho ba prior đơn giản: đều, tần suất chung (unigram), tần suất theo vị trí."""
    n_te = len(Cte)
    flat_te = Cte.reshape(n_te, -1)
    uni = torch.bincount(Ctr.reshape(-1), minlength=K).double() + 1
    uni /= uni.sum()
    pos = torch.stack([torch.bincount(Ctr.reshape(len(Ctr), -1)[:, j], minlength=K) for j in range(GRID * GRID)])
    pos = (pos.double() + 1) / (pos.sum(1, keepdim=True) + K)
    return (
        {
            "uniform": GRID * GRID * math.log2(K),
            "unigram": float(-torch.log2(uni[flat_te]).sum(1).mean()),
            "position": float(-torch.log2(pos[torch.arange(GRID * GRID), flat_te]).sum(1).mean()),
        },
        uni,
        pos,
    )


pc = W["cfg"]["prior"]
prior = Prior(K, pc["d"], pc["layers"])
prior.load_state_dict(unpack(W["prior"]))
prior.eval()
print(
    f"prior: {pc['layers']} khối Transformer, rộng {pc['d']}, {vn(sum(p.numel() for p in prior.parameters()), 0)} tham số"
)

C_tr = codes(vqm, X)
N_LIVE = R["prior"]["cfg"]["nllLive"]
base, p_uni, p_pos = baseline_bits(C_tr, C_te[:N_LIVE], K)
got = {**base, "transformer": prior.nll_bits(C_te[:N_LIVE].reshape(N_LIVE, -1))}
want = R["prior"]["seeds"][0]["testLive"]
for k in got:
    assert abs(got[k] - want[k]) < 0.05 * (1 if k != "transformer" else 4), f"{k}: khác với script"
NAMES = {"uniform": "đều (mọi mã như nhau)", "unigram": "tần suất chung của mã", "position": "tần suất theo từng vị trí",
         "transformer": "Transformer tự hồi quy"}  # fmt: skip
print(f"\n{'prior':28s}{f'bit/ảnh, {vn(N_LIVE, 0)} ảnh đầu':>24s}{'bit/ảnh, cả tập kiểm tra (3 seed)':>36s}")
for k, name in NAMES.items():
    full = [s["test"][k] for s in R["prior"]["seeds"]]
    print(f"{name:28s}{vn(got[k], 1):>24s}{span(full, 1):>36s}")
print(
    f"\nbớt so với prior đều: tần suất chung {vn(got['uniform'] - got['unigram'], 1)} bit, theo vị trí "
    f"{vn(got['uniform'] - got['position'], 1)} bit, Transformer {vn(got['uniform'] - got['transformer'], 1)} bit"
)
print(
    f"ảnh gốc {vn(bits_raw, 0)} bit / Transformer {vn(got['transformer'], 1)} bit = {vn(bits_raw / got['transformer'], 1)} lần"
)

Prior đều tốn đúng 343 bit, bằng 49 · log₂ 128: nó không biết gì. Biết tần suất chung của từng mã chỉ bớt 16,5 bit;
biết tần suất theo từng vị trí (góc ảnh hầu như là nền, giữa ảnh là vải) bớt 46,5 bit. Transformer chỉ cần khoảng
một nửa số bit của prior đều: khi đã thấy các token trước, token kế tiếp dễ đoán hơn nhiều. Nói theo nén: 6.272 bit
của ảnh gốc còn khoảng 170 bit, nhỏ đi 36,6 lần, nhưng là nén *có mất mát*: ảnh dựng lại mờ như ở mục 6.

### 🤔 Dự đoán trước
Prior "theo từng vị trí" biết, chẳng hạn, rằng góc ảnh hầu như luôn là mã nền còn giữa ảnh hay là mã của vải. Rút
mỗi token *độc lập* theo đúng tần suất ấy ở vị trí của nó rồi giải mã: ảnh ra có giống một món đồ không?

In [ ]:
@torch.no_grad()
def sample(prior, n, g, temp=1.0, prefix=None):
    """Rút token lần lượt theo hàng; prefix (n, t) là các token đầu đã cho sẵn (để điền tiếp ảnh)."""
    c = torch.zeros(n, 0, dtype=torch.long) if prefix is None else prefix.clone()
    while c.shape[1] < GRID * GRID:
        logits = prior(torch.cat([c, torch.zeros(n, 1, dtype=torch.long)], 1))[:, -1] / temp
        c = torch.cat([c, torch.multinomial(F.softmax(logits, -1), 1, generator=g)], 1)
    return c.view(n, GRID, GRID)


g = torch.Generator().manual_seed(0)
n = 12
draws = {
    "đều": torch.randint(K, (n, GRID, GRID), generator=g),
    "theo vị trí": torch.stack([torch.multinomial(p_pos[j].float(), n, True, generator=g) for j in range(49)], 1),
    "Transformer": sample(prior, n, g),
}
with torch.no_grad():
    rows = {k: vqm.decode(c.view(n, GRID, GRID)) for k, c in draws.items()}
    rows["ảnh thật (tái tạo)"] = vqm(T[:n])[0]
fig, axes = plt.subplots(4, 1, figsize=(10, 4.4))
for ax, (name, imgs) in zip(axes, rows.items(), strict=True):
    show_row(ax, imgs, name)
plt.tight_layout()
plt.show()

Mỗi hàng là 12 mẫu. **Đều:** không phải quần áo, chỉ là các mảng vá. **Theo vị trí:** lần nào cũng ra gần như cùng
một vệt, một "món đồ trung bình" có cả tay áo lẫn ống quần, vì mỗi ô được rút riêng và không ô nào biết ô bên cạnh
thuộc món gì. Tần suất ở từng vị trí đúng, nhưng sự phối hợp giữa các vị trí thì mất. **Transformer:** quần, áo,
túi, giày nhận ra được, dù mờ và méo hơn ảnh thật tái tạo ở hàng cuối. Bộ giải mã không đổi; chính prior biến bộ
token hóa thành một mô hình sinh.

### Điền tiếp một ảnh

Vì prior đọc token theo hàng, ta có thể cho trước vài hàng đầu và để nó viết tiếp, như cho một mô hình ngôn ngữ phần
đầu câu. Mỗi hàng của hình dưới là một ảnh kiểm tra: ảnh gốc, ảnh chỉ còn 2 hàng token đầu (14 trong 49 token; phần
còn lại thay bằng mã nền), rồi bốn lần điền tiếp khác nhau.

In [ ]:
KEEP = 2 * GRID
g = torch.Generator().manual_seed(1)
fig, axes = plt.subplots(len(pick[:6]), 1, figsize=(6.5, 6.4))
for ax, i in zip(axes, pick[:6].tolist(), strict=True):
    c = C_te[i].reshape(1, -1)
    cut = c.clone()
    cut[:, KEEP:] = bg
    done = sample(prior, 4, g, prefix=c[:, :KEEP].repeat(4, 1))
    with torch.no_grad():
        imgs = torch.cat([T[i : i + 1], vqm.decode(cut.view(1, GRID, GRID)), vqm.decode(done)])
    show_row(ax, imgs, CLASSES[int(y_te[i])])
axes[0].set_title("gốc · 2 hàng token đầu · bốn lần điền tiếp", fontsize=9)
plt.tight_layout()
plt.show()

Hai hàng token đầu chỉ là 8 hàng điểm ảnh trên cùng, và prior điền tiếp những gì hợp với chúng. Phần đầu của một
chiếc áo thun không tay giống phần cạp của quần hay vai của váy, nên các lần điền tiếp có thể ra những món khác nhau;
vai rộng của áo len và áo khoác thì cho ra áo có tay. Đôi dép thấp đến mức hai hàng đầu toàn là nền: prior chỉ biết
"phía trên trống", và điền ra một món giày dép, loại đồ có phía trên trống. Đây là đúng cách một mô hình ngôn ngữ
viết tiếp một câu, chỉ khác là "câu" ở đây là ảnh.

## 10 · Thử thay đổi

### Nhiệt độ khi sinh

Chia logit cho nhiệt độ T trước softmax (như Bài 17): T < 1 dồn xác suất về các token hay gặp, T > 1 trải đều ra.
Thử thay `NHIET_DO` rồi chạy lại.

In [ ]:
NHIET_DO = (0.6, 1.0, 1.4)
g = torch.Generator().manual_seed(2)
fig, axes = plt.subplots(len(NHIET_DO), 1, figsize=(10, 1.2 * len(NHIET_DO) + 0.4))
for ax, t in zip(np.atleast_1d(axes), NHIET_DO, strict=True):
    c = sample(prior, 12, g, temp=t)
    with torch.no_grad():
        show_row(ax, vqm.decode(c), f"T = {vn(t, 1)}")
    print(
        f"T = {vn(t, 1)}: {len(torch.unique(c))} mã khác nhau trong 12 ảnh, {pct(float(is_bg[c].float().mean()))} số ô là mã nền"
    )
plt.tight_layout()
plt.show()

### Huấn luyện lại với cấu hình khác

Ô dưới huấn luyện lại vài trăm bước với một cấu hình bạn chọn, rồi so với lần chạy đầy đủ của script ở cùng bước
(nếu cấu hình ấy có trong lưới). Thử `init="uniform"`, `restart=True` với `init="randn"`, `K=512`, `beta=0` hay
`update="ema"`.

In [ ]:
CAU_HINH = {**REF, "init": "randn", "restart": False}
BUOC, SEED = 200, 0
m_try = train(CAU_HINH, SEED, BUOC)
res = metrics(m_try, V)
print(
    f"{BUOC} bước: MSE kiểm định {vn(res['mse'], 5)}, {res.get('used', '—')} mã được dùng, {m_try.q.restarts} lần thay mã"
)
for name, run in R["runs"].items():
    if run[SEED]["cfg"] == CAU_HINH and str(BUOC) in run[SEED]["checks"]:
        s = run[SEED]["checks"][str(BUOC)]
        print(f"script ({name}, cùng bước): MSE {vn(s['mse'], 5)}, {int(s.get('used', 0))} mã; sau {vn(STEPS, 0)} bước: "
              f"{span(seeds(name), 4)} MSE, {span(seeds(name, 'used'), 0)} mã (3 seed)")  # fmt: skip

### Sửa token

Token là một "bản vẽ" rời rạc: đổi vài token là đổi một vùng ảnh. Ô dưới lấy chiếc quần ở mục 6 và thay cột token
giữa (cột 4, đánh số từ 1) bằng mã nền, để xem bộ giải mã vẽ gì vào chỗ trống. Thử đổi sang hàng khác, mã khác.

In [ ]:
i = int(pick[1])
c = C_te[i].clone()
c[:, 3] = bg
with torch.no_grad():
    imgs = torch.cat([T[i : i + 1], vqm.decode(C_te[i : i + 1]), vqm.decode(c[None])])
fig, ax = plt.subplots(figsize=(4, 1.6))
show_row(ax, imgs)
ax.set_title("gốc · tái tạo · cột token giữa thành mã nền", fontsize=9)
plt.show()

## 11 · Giới hạn và cầu nối

- **Nhỏ.** Ảnh xám 28 × 28, mạng 31.377 tham số, 3.000 bước; các mô hình thật dùng ảnh màu 256 × 256 trở lên, mạng
  hàng chục triệu tham số và hàng trăm nghìn bước. Kết luận ở đây là về *cơ chế*, không về con số tuyệt đối.
- **Chỉ MSE.** Khi không chắc một chi tiết nằm đâu, đoán trung bình của các khả năng cho MSE nhỏ nhất, nên mất mát
  bình phương ưu ái ảnh mờ (Bài 19). VQGAN thêm mất mát cảm nhận và một bộ phân biệt GAN trên từng mảng ảnh (Bài 20),
  và chính điều đó làm ảnh nét.
- **Một tầng, một lưới.** VQ-VAE-2 dùng hai lưới (32 × 32 và 64 × 64 cho ảnh 256 × 256); các bộ token hóa âm thanh
  chồng nhiều bảng mã lên nhau theo kiểu dư (RVQ, Chuyên đề 05).
- **Prior nhỏ, đọc theo hàng.** MaskGIT thay prior tự hồi quy bằng một Transformer hai chiều đoán nhiều token bị che
  cùng lúc, sinh ảnh trong 8 bước thay vì 256.
- **3 seed, một η.** η được chọn cho cấu hình chuẩn rồi dùng chung; một cấu hình khác (FSQ, β = 0) có thể tốt hơn
  ở η riêng của nó.

Những gì đã dựng ở đây là xương sống của nhiều hệ thống lớn. DALL·E (2021) nén ảnh 256 × 256 thành 32 × 32 token
(8.192 mã, nhưng học bằng Gumbel-softmax thay vì straight-through). Parti sinh token ViT-VQGAN từ một câu mô tả;
Chameleon đưa token ảnh vào cùng chuỗi với token chữ: bộ token hóa của nó biến ảnh 512 × 512 thành 1.024 token từ
bảng 8.192 mã. Latent Diffusion (Bài 21) dùng cùng loại bộ tự mã hóa, nhưng khuếch tán trên mã liên tục. Và mọi bộ
token hóa âm thanh của Chuyên đề 05 là VQ-VAE với nhiều tầng mã.

## Tóm tắt

- **Token hóa ảnh:** bộ mã hóa CNN biến ảnh 28 × 28 thành lưới 7 × 7 vectơ; mỗi vectơ thành chỉ số của mã gần nhất
  trong 128 mã. Một ảnh còn 49 token, 343 bit, nhỏ hơn ảnh gốc 18,3 lần.
- **Straight-through** chép gradient qua bước tìm mã gần nhất, nên bảng mã không nhận gradient nào từ tái tạo. Hai
  số hạng thêm vào không đối xứng: mất mát mã dạy bảng mã, cam kết (β = 0,25) dạy bộ mã hóa.
- **Bảng mã chết:** chỉ mã được chọn mới học. Khởi tạo N(0, 1) để lại vài mã dùng được và không tự sửa; khởi tạo
  từ dữ liệu cộng khởi động lại mã chết (Jukebox) dùng đủ 128 mã.
- **Mã là mẫu cục bộ**, không phải loại đồ; nhiều mã là bản sao của nền đen, nên "dùng đủ 128 mã" không có nghĩa là
  128 thứ khác nhau.
- **Các nhận định:** A gần như khớp (β 0,1–1 trong dao động, β 2 kém hơn chút); B khớp (β = 0 làm $z_e$ dài ra hơn
  mười nghìn lần); C không khớp ở đây (EMA ngang mất mát mã, nhưng tự cứu một phần khởi tạo tồi); D khớp
  (autoencoder liên tục cho MSE nhỏ hơn 2,38 lần); E khớp (K lớn tốt hơn, nhưng cần khởi động lại); F tùy điều
  kiện (FSQ dùng đủ 128 mã, nhưng chỉ một phần của 1.000 mã).
- **Sinh ảnh** cần một prior trên token. Một Transformer nhỏ (119.808 tham số) mã hóa một ảnh kiểm tra bằng
  168,4–172,7 bit, so với 343 của prior đều và 296,8–298,0 của prior theo vị trí; mẫu của nó nhận ra được, còn rút
  từng vị trí độc lập chỉ ra một vệt mờ. Nó cũng điền tiếp được một ảnh từ vài hàng token đầu.

## Câu hỏi ôn tập

1. Vì sao gradient của mất mát tái tạo đối với bảng mã bằng đúng 0? Nếu bỏ mất mát mã mà vẫn giữ cam kết, bảng mã
   có học được không?
2. Bảng mã N(0, 1) 16 chiều chỉ để lại vài mã được dùng, và huấn luyện không sửa được. Giải thích bằng độ dài của
   $z_e$ và của các mã. Hai cách sửa nào đã dùng ở đây?
3. Một bộ token hóa cho ảnh 256 × 256 dùng lưới 32 × 32 và 8.192 mã. Một ảnh bao nhiêu bit? Ảnh gốc 8 bit mỗi kênh
   màu bao nhiêu bit?
4. Mô hình chuẩn dùng đủ 128 mã, nhưng nhiều mã là bản sao của nền đen. Vì sao khởi động lại mã chết sinh ra bản
   sao? Perplexity cao hơn có luôn nghĩa là bảng mã tốt hơn không?
5. Rút mỗi token độc lập theo tần suất ở vị trí của nó cho ra gần như cùng một vệt mờ ở mọi lần. Vì sao? Prior
   Transformer biết thêm điều gì?
6. Autoencoder liên tục dựng lại tốt hơn VQ-VAE. Vậy vì sao các hệ thống như DALL·E, Parti hay Chameleon vẫn lượng
   tử hóa ảnh?

<details><summary>Gợi ý đáp án</summary>

1. Đầu vào của bộ giải mã là `ze + (zq − ze).detach()`: theo giá trị là `zq`, theo gradient là `ze`; `zq` chỉ nằm
   trong phần bị `.detach()`, nên không có đường nào từ mất mát tái tạo về bảng mã. Cam kết $\lVert z_e -
   \mathrm{sg}[e]\rVert^2$ cũng dừng gradient ở $e$. Bỏ mất mát mã thì bảng mã đứng yên ở giá trị khởi tạo (trừ khi
   dùng bản EMA, cập nhật mã không qua gradient).
2. Lúc khởi tạo, $z_e$ dài khoảng 0,4 và chụm gần gốc, còn các mã N(0, 1) dài khoảng 4. Mã gần nhất của hầu hết
   $z_e$ là cùng một hai mã tình cờ nằm gần gốc nhất. Các mã khác không được chọn nên không nhận gradient của mất mát
   mã, nên không bao giờ tiến lại gần $z_e$. Cách sửa: lấy chính các $z_e$ của batch đầu làm bảng mã, và thay mã
   không ai dùng bằng một $z_e$ ngẫu nhiên của batch hiện tại.
3. log₂ 8.192 = 13 bit, nên 32 · 32 · 13 = 13.312 bit. Ảnh gốc: 256 · 256 · 3 · 8 = 1.572.864 bit, gấp 118 lần.
   (Đó là cấu hình của DALL·E, dù DALL·E học bảng mã bằng Gumbel-softmax.)
4. Một mã chết được thay bằng một $z_e$ ngẫu nhiên của batch, và một nửa số điểm ảnh là nền đen, nên rất nhiều $z_e$
   là "nền". Mã mới vì thế thường rơi vào vùng nền, chia nhau những ô nền vốn chỉ cần một mã. Perplexity đo số mã
   được dùng đều nhau, không đo chúng có khác nhau hay không; Huh và cộng sự (2023) cũng cảnh báo rằng perplexity cao
   có thể chỉ là dư thừa.
5. Mỗi vị trí được rút riêng, nên token ở ô này không biết ô bên cạnh là áo hay là quần: ảnh ra là một trung bình
   chắp vá của mọi món đồ. Prior Transformer đọc các token trước, nên biết hai hàng đầu là cổ áo thì các hàng sau
   phải là thân áo.
6. Vì token cho hai thứ mà mã liên tục không có: số bit nhỏ, và một chuỗi số nguyên mà một Transformer học được bằng
   đúng cách học chữ (dự đoán token kế tiếp với entropy chéo), dùng chung một mô hình cho cả chữ lẫn ảnh. Chất lượng
   mất đi ở bước lượng tử hóa được bù bằng bộ giải mã mạnh hơn và các mất mát cảm nhận và GAN.
</details>

## Tìm hiểu thêm

- van den Oord, Vinyals & Kavukcuoglu (2017), *Neural Discrete Representation Learning* (VQ-VAE):
  [arXiv:1711.00937](https://arxiv.org/abs/1711.00937).
- Razavi, van den Oord & Vinyals (2019), *Generating Diverse High-Fidelity Images with VQ-VAE-2*:
  [arXiv:1906.00446](https://arxiv.org/abs/1906.00446).
- Dhariwal và cộng sự (2020), *Jukebox: A Generative Model for Music* (khởi động lại mã chết):
  [arXiv:2005.00341](https://arxiv.org/abs/2005.00341).
- Esser, Rombach & Ommer (2021), *Taming Transformers for High-Resolution Image Synthesis* (VQGAN):
  [arXiv:2012.09841](https://arxiv.org/abs/2012.09841).
- Yu và cộng sự (2021), *Vector-quantized Image Modeling with Improved VQGAN*:
  [arXiv:2110.04627](https://arxiv.org/abs/2110.04627).
- Chang và cộng sự (2022), *MaskGIT: Masked Generative Image Transformer*:
  [arXiv:2202.04200](https://arxiv.org/abs/2202.04200).
- Rombach và cộng sự (2022), *High-Resolution Image Synthesis with Latent Diffusion Models*:
  [arXiv:2112.10752](https://arxiv.org/abs/2112.10752).
- Huh và cộng sự (2023), *Straightening Out the Straight-Through Estimator: Overcoming Optimization Challenges in
  Vector Quantized Networks*: [arXiv:2305.08842](https://arxiv.org/abs/2305.08842).
- Mentzer và cộng sự (2023), *Finite Scalar Quantization: VQ-VAE Made Simple*:
  [arXiv:2309.15505](https://arxiv.org/abs/2309.15505).
- Fifty và cộng sự (2024), *Restructuring Vector Quantization with the Rotation Trick*:
  [arXiv:2410.06424](https://arxiv.org/abs/2410.06424).
- Chameleon Team (2024), *Chameleon: Mixed-Modal Early-Fusion Foundation Models*:
  [arXiv:2405.09818](https://arxiv.org/abs/2405.09818).
- NVIDIA (2025), *Cosmos World Foundation Model Platform for Physical AI* (bộ token hóa FSQ):
  [arXiv:2501.03575](https://arxiv.org/abs/2501.03575).
- Mã gốc của DeepMind: [`sonnet/src/nets/vqvae.py`](https://github.com/google-deepmind/sonnet/blob/v2/sonnet/src/nets/vqvae.py)
  và [`vqvae_example.ipynb`](https://github.com/google-deepmind/sonnet/blob/v2/examples/vqvae_example.ipynb).
- Xiao, Rasul & Vollgraf (2017), *Fashion-MNIST*: [arXiv:1708.07747](https://arxiv.org/abs/1708.07747).

**Quay lại:** [Bài 19 · Autoencoder & VAE](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/) ·
[Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) ·
[Chuyên đề 05 · Token hóa âm thanh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-hoa-am-thanh/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)